In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from sklearn.model_selection import train_test_split

## Create Model

In [ ]:
class Model(nn.Module):

    def __init__(self, in_features, h1, h2, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)

        return x

## Visualize Data

In [ ]:
df = pd.read_csv('C:\\Users\\alvar\\Documents\\torch_templates\\DATA\\iris.csv')
df

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10,7))
fig.tight_layout()

plots = [(0,1), (2,3), (0,2), (1,3)]
colors = ['b', 'r', 'g']
labels = ['Iris setosa', 'Iris virginica', 'Iris versicolor']

for i, ax in enumerate(axes.flat):
    for j in range(3):
        x = df.columns[plots[i][0]]
        y = df.columns[plots[i][1]]
        ax.scatter(df[df['target']==j][x], df[df['target']==j][y], color=colors[j])
        ax.set(xlabel=x, ylabel=y)

fig.legend(labels=labels, loc=3, bbox_to_anchor=(1.0, 0.85))
plt.show();

## Split Data and Train

In [ ]:
X = df.drop('target', axis=1).values
y = df['target'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

In [ ]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

In [ ]:
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [ ]:
torch.manual_seed(101)
model = Model(4, 4, 4, 3)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 500
losses = []

for i in range(1, epochs+1):

    y_pred = model.forward(X_train)

    loss = criterion(y_pred, y_train)

    losses.append(loss.detach().numpy())

    print(f'Epoch = {i}, loss = {loss}')

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
plt.plot(range(1,epochs+1), losses)
plt.xlabel('epochs'), plt.ylabel('loss');

## Validation

In [ ]:
with torch.no_grad():
    y_eval = model.forward(X_test)
    loss = criterion(y_eval, y_test)

In [ ]:
loss

In [ ]:
correct = 0

with torch.no_grad():

    for i, data in enumerate(X_test):
        y_val = model.forward(data)
        print(f'{i+1}:\t True: {y_val.argmax().item()}\t Predict: {y_test[i]}')
        
        if y_val.argmax().item() == y_test[i]:
            correct += 1

print(f'Correct predictions: {correct}/{len(X_test)}')

## Saving Model

In [ ]:
# torch.save(model.state_dict(), 'C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\iris_model.pt')
# new_model = Model(4,4,4,3)
# new_model.load_state_dict(torch.load('C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\iris_model.pt'))
# new_model.eval()

In [ ]:
torch.save(model, 'C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\iris_model.pt')

new_model = torch.load('C:\\Users\\alvar\\Documents\\torch_templates\\ANN\\iris_model.pt')
new_model.eval()

## Prediction

In [ ]:
my_iris = torch.tensor([5.6, 3.7, 2.2, 0.5])

In [ ]:
with torch.no_grad():
    print(new_model(my_iris).argmax())